# Thyroid Disease (ann-thyroid) — Combine, Label Columns, Split, Export

Self-contained data preparation notebook. Assumes `ann-train.data` and
`ann-test.data` are already sitting in `data/raw/` (no downloading here).

Steps:
1. Load both raw files from `data/raw/`
2. Add the 21 documented **column** names (the class itself stays numeric — see note below)
3. Combine both files into one dataset
4. Re-split 80% train / 20% test (stratified on class)
5. Export `data/processed/train.csv` and `data/processed/test.csv`

**On the class column**: the raw data already encodes it as `1` / `2` / `3`
(normal / hyperthyroid / hypothyroid) — that's already a clean numeric label,
so we keep it as-is instead of converting to text and re-encoding it later.
A separate `class_labels.json` is exported just for readable plots/reports;
it's never used for training.


## 0. Setup

In [6]:
import os
import json

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20

RAW_DIR = "data/raw"
OUTPUT_DIR = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(RAW_DIR, "ann-train.data")
TEST_PATH = os.path.join(RAW_DIR, "ann-test.data")

assert os.path.exists(TRAIN_PATH), f"Couldn't find {TRAIN_PATH} — check the filename/path in data/raw/"
assert os.path.exists(TEST_PATH), f"Couldn't find {TEST_PATH} — check the filename/path in data/raw/"


## 1. Load and add column names

These files are whitespace-delimited with no header row. Documented column
order: 15 binary flags, then 6 continuous measurements, then the class label
(1 = normal, 2 = hyperthyroid, 3 = hypothyroid — already numeric, kept as-is).


In [7]:
COLUMN_NAMES = [
    "age", "sex", "on_thyroxine", "query_on_thyroxine",
    "on_antithyroid_medication", "sick", "pregnant", "thyroid_surgery",
    "I131_treatment", "query_hypothyroid", "query_hyperthyroid", "lithium",
    "goitre", "tumor", "hypopituitary", "psych",
    "TSH", "T3", "TT4", "T4U", "FTI",
    "class",
]

def load_ann_file(path):
    df = pd.read_csv(path, sep=r"\s+", header=None, engine="python")
    df = df.dropna(how="all")  # drop any fully-empty trailing rows

    n_expected = len(COLUMN_NAMES)
    if df.shape[1] > n_expected:
        df = df.iloc[:, :n_expected]  # drop any extra trailing field some copies include
    df.columns = COLUMN_NAMES[: df.shape[1]]

    # class can look like "3.|1234" in some distributions, or a plain "3" in others
    df["class"] = df["class"].astype(str).str.extract(r"(\d+)").astype(int)
    return df.reset_index(drop=True)

train_raw = load_ann_file(TRAIN_PATH)
test_raw = load_ann_file(TEST_PATH)

print("ann-train rows:", len(train_raw))
print("ann-test rows:", len(test_raw))
train_raw.head()


ann-train rows: 3772
ann-test rows: 3428


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,...,goitre,tumor,hypopituitary,psych,TSH,T3,TT4,T4U,FTI,class
0,0.73,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0.00060,0.015,0.120,0.082,0.146,3
1,0.24,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00025,0.030,0.143,0.133,0.108,3
2,0.47,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00190,0.024,0.102,0.131,0.078,3
3,0.64,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00090,0.017,0.077,0.090,0.085,3
4,0.23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00025,0.026,0.139,0.090,0.153,3


## 2. Combine both files

In [8]:
combined = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)
print("Combined rows:", len(combined))
print(combined["class"].value_counts().sort_index())

CLASS_LABELS = {1: "hyperthyroid", 2: "hypothyroid", 3: "normal"}  # for reporting only, not used in training
# Verified against class 3 being ~92% of rows, matching the documented "92% not hyperthyroid" fact for this dataset.


Combined rows: 7200
class
1     166
2     368
3    6666
Name: count, dtype: int64


## 3. 80/20 train/test split

Stratified on `class` so both splits keep the same ~92/6/2 class balance —
otherwise a random split could easily leave the test set with very few
hyperthyroid/hypothyroid examples.


In [9]:
train_df, test_df = train_test_split(
    combined,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=combined["class"],
)

print(f"Train: {len(train_df)} rows ({len(train_df)/len(combined):.0%})")
print(f"Test:  {len(test_df)} rows ({len(test_df)/len(combined):.0%})")
print()
print("Train class balance:")
print(train_df["class"].value_counts(normalize=True).sort_index())
print()
print("Test class balance:")
print(test_df["class"].value_counts(normalize=True).sort_index())


Train: 5760 rows (80%)
Test:  1440 rows (20%)

Train class balance:
class
1    0.023090
2    0.051042
3    0.925868
Name: proportion, dtype: float64

Test class balance:
class
1    0.022917
2    0.051389
3    0.925694
Name: proportion, dtype: float64


## 4. Export as CSV (with column headers) + the label legend

In [10]:
train_out_path = os.path.join(OUTPUT_DIR, "train.csv")
test_out_path = os.path.join(OUTPUT_DIR, "test.csv")
labels_out_path = os.path.join(OUTPUT_DIR, "class_labels.json")

train_df.to_csv(train_out_path, index=False)
test_df.to_csv(test_out_path, index=False)

with open(labels_out_path, "w") as f:
    json.dump(CLASS_LABELS, f, indent=2)

print("Saved:")
print(" -", train_out_path, train_df.shape)
print(" -", test_out_path, test_df.shape)
print(" -", labels_out_path, "(readable class names, for reports/plots only)")


Saved:
 - data/processed\train.csv (5760, 22)
 - data/processed\test.csv (1440, 22)
 - data/processed\class_labels.json (readable class names, for reports/plots only)


### Next steps

`data/processed/train.csv` and `data/processed/test.csv` keep the numeric
`class` column (`1`/`2`/`3`) exactly as the raw data has it — no unnecessary
text round-trip. `class_labels.json` is there purely so evaluation code can
show `"normal"` instead of `1` in confusion matrices / reports.

Both the Random Forest notebook and the DL notebook should load these two
files directly and apply whatever feature scaling they individually need
(RF doesn't need scaling, a neural net does).
